## Setup and Installation

In [ ]:
# Install Java (required for Pyserini)
!apt-get update
!apt-get install -y openjdk-21-jdk
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio
!pip install pyserini==0.36.0
!pip install transformers
!pip install pandas numpy

In [ ]:
# Hugging Face Authentication
from huggingface_hub import login

hugging_face_token = "YOUR_HF_TOKEN_HERE"
login(hugging_face_token)

## Load Fixed Components (Cannot Change)

In [ ]:
import pandas as pd
import json
import re
import string
from collections import Counter
import numpy as np

# Load the FIXED Pyserini index (wikipedia-kilt-doc)
from pyserini.search import SimpleSearcher
from pyserini.index.lucene import IndexReader

searcher = SimpleSearcher.from_prebuilt_index('wikipedia-kilt-doc')
index_reader = IndexReader.from_prebuilt_index('wikipedia-kilt-doc')

print("Index loaded (FIXED - cannot change):")
print(index_reader.stats())

In [ ]:
# Mount Google Drive and load data
from google.colab import drive
drive.mount('/content/drive')

# Update these paths to match your Google Drive structure
train_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/train.csv"
test_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/test.csv"

# CRITICAL: Use converters to parse answers as JSON
df_train = pd.read_csv(train_path, converters={"answers": json.loads})
df_test = pd.read_csv(test_path)

print(f"Train: {len(df_train)} questions")
print(f"Test: {len(df_test)} questions")
print("\nSample:")
print(df_train.head(3))

In [ ]:
# Load the FIXED LLM (Llama-3.2-1B-Instruct)
import transformers
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"  # FIXED - cannot change

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("LLM loaded (FIXED - cannot change):", model_id)

## Optimized Retrieval Functions (What We CAN Improve!)

In [ ]:
# Strategy 1: Advanced Query Preprocessing
def preprocess_query(query):
    """
    Aggressively preprocess query to match document vocabulary.
    This is KEY to improving retrieval!
    """
    query = query.lower().strip()
    
    # Remove question patterns that don't appear in documents
    query = re.sub(r'^what is the name of ', '', query)
    query = re.sub(r'^what is the ', '', query)
    query = re.sub(r'^what are the ', '', query)
    query = re.sub(r'^what does ', '', query)
    query = re.sub(r'^what did ', '', query)
    query = re.sub(r'^where is ', '', query)
    query = re.sub(r'^where are ', '', query)
    query = re.sub(r'^where did ', '', query)
    query = re.sub(r'^where was ', '', query)
    query = re.sub(r'^when is ', '', query)
    query = re.sub(r'^when did ', '', query)
    query = re.sub(r'^when was ', '', query)
    query = re.sub(r'^who is ', '', query)
    query = re.sub(r'^who are ', '', query)
    query = re.sub(r'^who did ', '', query)
    query = re.sub(r'^who was ', '', query)
    query = re.sub(r'^who does ', '', query)
    query = re.sub(r'^which ', '', query)
    query = re.sub(r'^how many ', '', query)
    query = re.sub(r'^how much ', '', query)
    
    # Remove trailing question marks
    query = query.rstrip('?').strip()
    
    # Clean whitespace
    query = ' '.join(query.split())
    
    return query


def expand_query(query):
    """
    Generate multiple query variations for better retrieval coverage.
    """
    queries = []
    
    # Original processed query
    processed = preprocess_query(query)
    queries.append(processed)
    
    # Remove common stop words for keyword-based search
    stop_words = {
        'what', 'where', 'when', 'who', 'which', 'how', 'why',
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'does', 'do', 'did', 'done', 'doing',
        'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'from', 'with', 'by'
    }
    
    tokens = processed.split()
    keywords = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    if keywords and len(keywords) < len(tokens):
        keyword_query = ' '.join(keywords)
        queries.append(keyword_query)
    
    # If we have multiple keywords, try bigrams (pairs of words)
    if len(keywords) >= 3:
        # Take most important keywords (usually nouns at the end)
        important = ' '.join(keywords[-3:])
        if important not in queries:
            queries.append(important)
    
    return queries


# Test query preprocessing
test_queries = [
    "what is the name of justin bieber brother?",
    "where did mozart write his four horn concertos for?",
    "who plays ken barlow in coronation street?"
]

print("Query Expansion Examples:\n")
for q in test_queries:
    expanded = expand_query(q)
    print(f"Original: {q}")
    for i, eq in enumerate(expanded, 1):
        print(f"  Variant {i}: {eq}")
    print()

In [ ]:
# Strategy 2: Multi-Query Retrieval with Score Fusion
def retrieve_with_fusion(query, k=10, mu=1000, boost_factor=0.5):
    """
    Retrieve documents using multiple query variations and fuse scores.
    
    Args:
        query: Original question
        k: Number of documents to retrieve per query variation
        mu: BM25 Dirichlet smoothing parameter
        boost_factor: Score boost for documents matching multiple queries
    
    Returns:
        List of top documents with content and scores
    """
    # Set BM25 parameter (THIS WE CAN TUNE!)
    searcher.set_qld(mu=mu)
    
    # Expand query into variations
    query_variations = expand_query(query)
    
    # Retrieve with each variation
    doc_scores = {}  # doc_id -> score
    doc_info = {}    # doc_id -> {content, title, hit_count}
    
    for q_var in query_variations:
        hits = searcher.search(q_var, k)
        
        for hit in hits:
            doc_id = hit.docid
            
            if doc_id not in doc_scores:
                # First time seeing this document
                doc = searcher.doc(doc_id)
                data = json.loads(doc.raw())
                
                doc_scores[doc_id] = hit.score
                doc_info[doc_id] = {
                    'content': data.get('contents', ''),
                    'title': data.get('title', ''),
                    'hit_count': 1
                }
            else:
                # Document appeared in multiple query results - BOOST IT!
                doc_scores[doc_id] += hit.score * boost_factor
                doc_info[doc_id]['hit_count'] += 1
    
    # Sort by final score
    sorted_doc_ids = sorted(doc_scores.keys(), key=lambda x: doc_scores[x], reverse=True)
    
    # Return top k documents
    results = []
    for doc_id in sorted_doc_ids[:k]:
        info = doc_info[doc_id]
        results.append({
            'content': info['content'],
            'title': info['title'],
            'score': doc_scores[doc_id],
            'hit_count': info['hit_count']
        })
    
    return results

In [ ]:
# Strategy 3: Smart Context Selection
def format_contexts(retrieved_docs, max_length=600, include_title=True):
    """
    Format retrieved documents into context strings.
    
    Args:
        retrieved_docs: List of document dicts from retrieve_with_fusion
        max_length: Maximum characters per document
        include_title: Whether to prepend document title
    
    Returns:
        List of formatted context strings
    """
    contexts = []
    
    for doc in retrieved_docs:
        content = doc['content'].replace('\n', ' ').strip()
        
        # Prepend title if available and requested
        if include_title and doc['title']:
            context = f"{doc['title']}: {content}"
        else:
            context = content
        
        # Truncate to max length
        if len(context) > max_length:
            context = context[:max_length]
        
        contexts.append(context)
    
    return contexts

## Optimized Prompting (Can Improve!)

In [ ]:
def create_optimized_prompt(question, contexts):
    """
    Create an optimized prompt for Llama-3.2-1B-Instruct.
    
    Small models work best with:
    - Very clear, direct instructions
    - Simple language
    - Explicit formatting requirements
    """
    # Join contexts with clear separators
    context_text = '\n---\n'.join([
        f"Passage {i+1}: {ctx}" 
        for i, ctx in enumerate(contexts)
    ])
    
    system_message = (
        "You are a helpful assistant that extracts precise answers from text passages. "
        "Your answer must be:"
        "\n1. As SHORT as possible (ideally 1-3 words)"
        "\n2. Taken ONLY from the passages below"
        "\n3. WITHOUT any explanation or extra words"
        "\n4. If not found in passages, answer 'unknown'"
    )
    
    user_message = (
        f"Passages:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Extract the shortest answer from the passages above. "
        f"Answer only with the specific name, date, place, or fact - nothing else.\n\n"
        f"Answer:"
    )
    
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]
    
    return messages

In [ ]:
# Answer Post-Processing
def clean_answer(answer):
    """
    Clean LLM output to extract just the answer.
    """
    # Remove common prefixes
    prefixes = [
        'the answer is ',
        'answer: ',
        'according to ',
        'based on ',
        'from passage ',
        'the ',
    ]
    
    answer_lower = answer.lower().strip()
    for prefix in prefixes:
        if answer_lower.startswith(prefix):
            answer = answer[len(prefix):].strip()
            answer_lower = answer.lower().strip()
    
    # Remove quotes
    answer = answer.strip('"\"'').strip()
    
    # Remove trailing punctuation (except abbreviations)
    if answer.endswith('.') and len(answer.split()) > 1:
        answer = answer[:-1].strip()
    
    # Handle unknown responses
    if answer_lower in ['unknown', 'i dont know', "i don't know", 'not found', 'not mentioned']:
        return 'unknown'
    
    # If too long, take first sentence
    if len(answer.split()) > 20:
        sentences = answer.split('.')
        if sentences:
            answer = sentences[0].strip()
    
    return answer.strip()

## Main Pipeline

In [ ]:
def answer_question(question, k=10, mu=1000, max_context_length=600, temperature=0.2):
    """
    Complete RAG pipeline with optimized retrieval.
    
    Args:
        question: Question to answer
        k: Number of documents to retrieve
        mu: BM25 smoothing parameter (can tune!)
        max_context_length: Max chars per document
        temperature: LLM temperature (lower = more focused)
    
    Returns:
        Answer string
    """
    # Step 1: Optimized Retrieval (THIS IS WHERE WE IMPROVE!)
    retrieved_docs = retrieve_with_fusion(question, k=k, mu=mu)
    
    if not retrieved_docs:
        return 'unknown'
    
    # Step 2: Format contexts
    contexts = format_contexts(retrieved_docs, max_length=max_context_length)
    
    # Step 3: Create prompt
    messages = create_optimized_prompt(question, contexts)
    
    # Step 4: Generate answer (using FIXED LLM)
    try:
        outputs = pipeline(
            messages,
            max_new_tokens=80,
            eos_token_id=terminators,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
        
        raw_answer = outputs[0]["generated_text"][-1].get('content', 'unknown')
        
        # Step 5: Post-process
        final_answer = clean_answer(raw_answer)
        
        return final_answer
        
    except Exception as e:
        print(f"Error: {e}")
        return 'unknown'

## Test on Examples

In [ ]:
# Test on a few training examples
print("Testing optimized retrieval on sample questions:\n")
print("=" * 80)

test_samples = df_train.sample(n=5, random_state=42)

for idx, row in test_samples.iterrows():
    question = row['question']
    ground_truth = row['answers']
    
    answer = answer_question(question, k=10, mu=1000)
    
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"Ground Truth: {ground_truth}")
    print("-" * 80)

## Evaluation Functions

In [ ]:
def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    """Compute token-level F1."""
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return int(pred_tokens == gt_tokens)
    if num_same == 0:
        return 0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def evaluate(predictions_dict, df_gold):
    """Calculate average F1 score."""
    f1_sum = 0.0
    count = 0
    
    for idx, row in df_gold.iterrows():
        qid = row['id']
        if qid not in predictions_dict:
            continue
        
        ground_truths = row['answers']
        prediction = predictions_dict[qid]
        
        # Max F1 over all ground truths
        f1 = max(f1_score(prediction, gt) for gt in ground_truths)
        f1_sum += f1
        count += 1
    
    return (100.0 * f1_sum / count) if count > 0 else 0.0

## Hyperparameter Tuning (Focus on Retrieval!)

In [ ]:
# Test different retrieval parameters
sample_size = 50  # Start small, increase if you have time
df_sample = df_train.sample(n=sample_size, random_state=42)

print(f"Tuning retrieval parameters on {sample_size} training examples...\n")

# Parameter configurations to test
configs = [
    {'k': 5, 'mu': 1000, 'temp': 0.2, 'max_ctx': 500},
    {'k': 10, 'mu': 1000, 'temp': 0.2, 'max_ctx': 600},  # Baseline
    {'k': 15, 'mu': 1000, 'temp': 0.2, 'max_ctx': 600},
    {'k': 10, 'mu': 500, 'temp': 0.2, 'max_ctx': 600},
    {'k': 10, 'mu': 2000, 'temp': 0.2, 'max_ctx': 600},
    {'k': 10, 'mu': 1000, 'temp': 0.1, 'max_ctx': 700},
]

best_score = 0
best_config = None

for i, config in enumerate(configs):
    print(f"\nConfig {i+1}/{len(configs)}: {config}")
    
    predictions = {}
    for idx, row in df_sample.iterrows():
        answer = answer_question(
            row['question'],
            k=config['k'],
            mu=config['mu'],
            max_context_length=config['max_ctx'],
            temperature=config['temp']
        )
        predictions[row['id']] = answer
    
    score = evaluate(predictions, df_sample)
    print(f"F1 Score: {score:.2f}%")
    
    if score > best_score:
        best_score = score
        best_config = config
        print("  ⭐ New best!")

print("\n" + "=" * 80)
print(f"Best Configuration: {best_config}")
print(f"Best F1 Score: {best_score:.2f}%")
print("=" * 80)

## Generate Test Predictions

In [ ]:
# Use best parameters from tuning (or set manually)
FINAL_K = 10
FINAL_MU = 1000
FINAL_TEMP = 0.2
FINAL_MAX_CTX = 600

print(f"Generating predictions for {len(df_test)} test questions...")
print(f"Parameters: k={FINAL_K}, mu={FINAL_MU}, temp={FINAL_TEMP}, max_ctx={FINAL_MAX_CTX}\n")

test_predictions = {}

for idx, row in df_test.iterrows():
    question = row['question']
    qid = row['id']
    
    answer = answer_question(
        question,
        k=FINAL_K,
        mu=FINAL_MU,
        max_context_length=FINAL_MAX_CTX,
        temperature=FINAL_TEMP
    )
    
    test_predictions[qid] = answer
    
    # Progress indicator
    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{len(df_test)}")
    
    # Show first few examples
    if idx < 3:
        print(f"\nQ: {question}")
        print(f"A: {answer}")

print("\n✓ Generation complete!")

## Save Predictions (CRITICAL FORMAT!)

In [ ]:
# Create submission DataFrame
submission_df = pd.DataFrame([
    {'id': qid, 'prediction': answer}
    for qid, answer in test_predictions.items()
])

# Sort by ID
submission_df = submission_df.sort_values('id').reset_index(drop=True)

# CRITICAL: Format predictions as JSON with ensure_ascii=False
submission_df['prediction'] = submission_df['prediction'].apply(
    lambda x: json.dumps([x], ensure_ascii=False)
)

# Verification
print("Verification:")
print(f"  Total predictions: {len(submission_df)}")
print(f"  Expected: {len(df_test)}")
print(f"  Match: {len(submission_df) == len(df_test)}")
print("\nFirst 3 predictions:")
print(submission_df.head(3))
print("\nFormat check:")
print(f"  Type: {type(submission_df.iloc[0]['prediction'])}")
print(f"  Value: {submission_df.iloc[0]['prediction']}")
print(f"  Can parse: {json.loads(submission_df.iloc[0]['prediction'])}")

# Save
output_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/optimized_retrieval_predictions.csv"
submission_df.to_csv(output_path, index=False)

print(f"\n✓ Saved to: {output_path}")